# 01. Train YOLOv10 sur Mac (`mps`) [v2 - regime principal SGD]

Notebook d'entraînement local pour une reproduction réduite de **YOLOv10**
sur **Pascal VOC** avec :

- `mps` au lieu de `cuda`
- un régime principal unique
- plus d'epochs
- `SGD` plus proche du papier

Exécuter ce notebook avec **Run All**. Il sauvegarde pour chaque run :

- `best.pt`
- `last.pt`
- `results.csv`
- un registre JSON partagé avec le notebook de visualisation

## Dépendances

Si besoin, installer une seule fois les dépendances du projet :

```python
# import sys, subprocess
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
```

In [ ]:
import os
import json
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO
from ultralytics import settings as ul_settings

def find_root() -> Path:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for candidate in candidates:
        if (candidate / "requirements.txt").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Impossible de trouver la racine du projet.")

ROOT = find_root()
os.chdir(ROOT)

RESULTS = ROOT / "results" / "mps_runs"
ANALYSIS = RESULTS / "analysis"
DATASETS_DIR = ROOT / "data" / "datasets"
REGISTRY_PATH = RESULTS / "experiment_registry.json"

for path in [RESULTS, ANALYSIS, DATASETS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
ul_settings.update({"datasets_dir": str(DATASETS_DIR), "runs_dir": str(RESULTS)})

SEED = 42
DATA_YAML = "VOC.yaml"
IMGSZ = 416
BATCH = 8
LR0 = 0.01
LRF = 0.01
WARMUP_EPOCHS = 3.0
MOMENTUM = 0.937
WEIGHT_DECAY = 5e-4
OPTIMIZER = "SGD"
WORKERS = 0
RUN_TAG = "mps_sgd_main"

REGIMES = {
    "main": {"epochs": 40, "fraction": 0.20},
}

MODELS = {
    "v10": "yolov10n.pt",
    "v8": "yolov8n.pt",
}

print({
    "root": str(ROOT),
    "device": DEVICE,
    "datasets_dir": str(DATASETS_DIR),
    "results_dir": str(RESULTS),
    "regimes": REGIMES,
})

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available() and hasattr(torch, "mps") and hasattr(torch.mps, "manual_seed"):
        torch.mps.manual_seed(seed)

def save_json(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def load_train_csv(path: Path):
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df

def latest_train_metrics(csv_path: Path) -> dict:
    df = load_train_csv(csv_path)
    if df is None or df.empty:
        return {}
    row = df.iloc[-1].to_dict()
    keep = {}
    for key, value in row.items():
        if any(tag in key for tag in ["mAP50", "mAP50-95", "precision", "recall", "epoch"]):
            try:
                keep[key] = float(value)
            except Exception:
                keep[key] = value
    return keep

def train_if_needed(model_name: str, regime_name: str, regime_cfg: dict) -> dict:
    run_name = f"{Path(model_name).stem}_{regime_name}_{RUN_TAG}"
    run_dir = RESULTS / run_name
    best = run_dir / "weights" / "best.pt"
    last = run_dir / "weights" / "last.pt"
    csv_path = run_dir / "results.csv"

    if best.exists() and last.exists():
        print(f"[skip] {run_name}")
    else:
        print(f"[train] {run_name} on {DEVICE}")
        set_seed(SEED)
        model = YOLO(model_name)
        model.train(
            data=DATA_YAML,
            epochs=regime_cfg["epochs"],
            fraction=regime_cfg["fraction"],
            imgsz=IMGSZ,
            batch=BATCH,
            device=DEVICE,
            optimizer=OPTIMIZER,
            lr0=LR0,
            lrf=LRF,
            cos_lr=True,
            warmup_epochs=WARMUP_EPOCHS,
            momentum=MOMENTUM,
            weight_decay=WEIGHT_DECAY,
            workers=WORKERS,
            cache=False,
            amp=False,
            pretrained=True,
            seed=SEED,
            deterministic=True,
            patience=100,
            project=str(RESULTS),
            name=run_name,
            exist_ok=True,
            save=True,
            plots=True,
            verbose=False,
        )

    if not best.exists() or not last.exists():
        raise FileNotFoundError(f"Run incomplet pour {run_name}.")

    return {
        "run_name": run_name,
        "model_key": "v10" if "yolov10" in model_name else "v8",
        "source_weights": model_name,
        "regime": regime_name,
        "epochs": regime_cfg["epochs"],
        "fraction": regime_cfg["fraction"],
        "device": DEVICE,
        "optimizer": OPTIMIZER,
        "lr0": LR0,
        "lrf": LRF,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "best": str(best),
        "last": str(last),
        "results_csv": str(csv_path),
        "metrics_from_csv": latest_train_metrics(csv_path),
    }

In [ ]:
registry = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "root": str(ROOT),
    "results_dir": str(RESULTS),
    "analysis_dir": str(ANALYSIS),
    "datasets_dir": str(DATASETS_DIR),
    "data_yaml": DATA_YAML,
    "device": DEVICE,
    "seed": SEED,
    "config": {
        "imgsz": IMGSZ,
        "batch": BATCH,
        "lr0": LR0,
        "lrf": LRF,
        "warmup_epochs": WARMUP_EPOCHS,
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "optimizer": OPTIMIZER,
        "workers": WORKERS,
        "run_tag": RUN_TAG,
    },
    "regimes": REGIMES,
    "runs": {},
}

if REGISTRY_PATH.exists():
    registry.update(load_json(REGISTRY_PATH))

for regime_name, regime_cfg in REGIMES.items():
    for model_key, model_name in MODELS.items():
        run_key = f"{model_key}_{regime_name}"
        registry["runs"][run_key] = train_if_needed(model_name, regime_name, regime_cfg)
        save_json(registry, REGISTRY_PATH)

print(f"Registre sauvegardé dans: {REGISTRY_PATH}")

In [ ]:
rows = []
for run_key, info in registry["runs"].items():
    rows.append(
        {
            "run_key": run_key,
            "run_name": info["run_name"],
            "model": info["model_key"],
            "regime": info["regime"],
            "epochs": info["epochs"],
            "fraction": info["fraction"],
            "best_exists": Path(info["best"]).exists(),
            "last_exists": Path(info["last"]).exists(),
            "best": info["best"],
            "last": info["last"],
        }
    )

summary_df = pd.DataFrame(rows).sort_values(["regime", "model"]).reset_index(drop=True)
summary_df

## Suite

Quand l'entraînement est terminé, ouvrir :

- `notebooks/02_viz_compare_yolov10_mps.ipynb`

Ce second notebook recharge automatiquement le registre, les checkpoints
`best` ou `last`, et génère les comparaisons et visualisations.